# 300-Step Benchmark Analysis: CPU (c23mm) vs GPU (c23g)

This notebook presents the benchmark results for 300 solver time steps across all 4 ML interface provider modes:
- **AIX** (LibTorch C++ backend)
- **SmartSim** (SmartRedis + Torch backend)
- **PhyDLL (C++)** (Native C++ DL client)
- **PhyDLL (Python)** (Python DL client via `mpi4py` + Torch)

## Hardware Setup & Allocations
- **CPU Runs:** 1 Node (`c23mm`, exclusive, 96 cores, 128GB+ RAM)
- **GPU Runs:** HetJob: 1 Node (`c23mm`, 24 solver ranks) + 1 Node (`c23g`, 1 NVIDIA GPU + 24 cores)
- **Slurm Account:** `thes2181`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load benchmark results
df = pd.read_csv('benchmark_results_300.csv')
df

In [ ]:
# Plot CPU vs GPU Solver Wall Times
providers = df['Provider'].unique()
cpu_times = [df[(df['Provider'] == p) & (df['Device'] == 'CPU')]['Solver Wall Time (s)'].values[0] for p in providers]
gpu_times = [df[(df['Provider'] == p) & (df['Device'] == 'GPU')]['Solver Wall Time (s)'].values[0] for p in providers]

x = np.arange(len(providers))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, cpu_times, width, label='CPU (c23mm)', color='#3498db')
rects2 = ax.bar(x + width/2, gpu_times, width, label='GPU (c23g)', color='#2ecc71')

ax.set_ylabel('Solver Wall Time (seconds)')
ax.set_title('300-Step Solver Wall Time Comparison (CPU vs GPU)')
ax.set_xticks(x)
ax.set_xticklabels(providers, rotation=15)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.7)

ax.bar_label(rects1, padding=3, fmt='%.0fs')
ax.bar_label(rects2, padding=3, fmt='%.0fs')

fig.tight_layout()
plt.savefig('benchmark_comparison_300.png', dpi=300)
plt.show()

## Key Summary Findings

1. **GPU Offloading Performance:**
   - **PhyDLL (C++) GPU** is the fastest overall solver configuration at **275 seconds** (~1.97x speedup over CPU).
   - **SmartSim GPU** follows closely at **287 seconds** (~1.84x speedup over CPU).
   - **PhyDLL (Python) GPU** achieves **321 seconds** (~1.80x speedup over CPU).

2. **AIX Scaling Note:**
   - **AIX** shows no GPU speedup over CPU in this configuration (**645s CPU vs 815s GPU**), indicating host-side serialization / memory transfer overhead in the AIxelerator service layer that warrants further profiling.

3. **PhyDLL Language Client Comparison:**
   - Native C++ client is **~35s faster on CPU** and **~46s faster on GPU** compared to the Python client (`mpi4py`), demonstrating lower IPC overhead.